# 03 — Q1: Unit Economics

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.db import get_conn, register_csv_tables, materialize_sql_file
from src.wise_theme import (
    apply as apply_wise_theme,
    BRIGHT_GREEN, DARK_GREEN, MID_GREEN, SOFT_GREEN, LIGHT_BG, WARM_GREY,
    diverging_cmap, sequential_cmap, style_axes, value_color, label_color_on,
)

FIG = ROOT / "figures"
FIG.mkdir(exist_ok=True)
apply_wise_theme()

con = get_conn()
register_csv_tables(con)
materialize_sql_file(con, ROOT / "sql" / "01_clean_transactions.sql")
materialize_sql_file(con, ROOT / "sql" / "02_cost_lookup.sql")
pp = materialize_sql_file(con, ROOT / "sql" / "03_profit_per_transaction.sql")
print(f"profit_per_transaction: {len(pp):,} rows × {pp.shape[1]} cols")

## 1. Headline — average profit per transaction type

In [ ]:
headline = (pp.groupby("transaction_type", as_index=False)
              .agg(n=("id", "count"),
                   avg_amount_gbp        = ("billing_in_gbp", "mean"),
                   avg_ic_revenue_gbp    = ("interchange_revenue_gbp", "mean"),
                   avg_conv_revenue_gbp  = ("conversion_revenue_gbp", "mean"),
                   avg_fixed_cost_gbp    = ("fixed_cost_gbp", "mean"),
                   avg_variable_cost_gbp = ("variable_cost_gbp", "mean"),
                   avg_profit_gbp        = ("profit_gbp", "mean"),
                   total_profit_gbp      = ("profit_gbp", "sum"))
              .sort_values("avg_profit_gbp", ascending=False))
headline.round(4)

In [ ]:
by_type = con.execute('''
    SELECT transaction_type, COUNT(*) AS n,
           AVG(profit_gbp) AS avg_profit,
           SUM(profit_gbp) AS total_profit
    FROM profit_per_transaction
    GROUP BY transaction_type
''').df()
order = ["ECOM_PURCHASE", "POS_PURCHASE", "CASH_WITHDRAWAL"]
by_type = by_type.set_index("transaction_type").reindex(order).reset_index()

fig, ax = plt.subplots(figsize=(8.0, 4.4))
x = np.arange(len(by_type))
colors = [value_color(v) for v in by_type["avg_profit"]]
ax.bar(x, by_type["avg_profit"], width=0.62, color=colors, edgecolor="none", zorder=3)
for i, v in enumerate(by_type["avg_profit"]):
    offset = 0.05 if v > 0 else -0.05
    va = "bottom" if v > 0 else "top"
    ax.text(i, v + offset, f"£{v:+.2f}", ha="center", va=va,
            fontweight="bold", fontsize=13, color=DARK_GREEN)
style_axes(ax, baseline=True)
ax.set_xticks(x); ax.set_xticklabels(by_type["transaction_type"], fontsize=11, fontweight="bold")
ax.set_ylabel("Avg profit per transaction (£)")
ax.set_title("Avg profit by transaction type")
ymax = max(abs(by_type["avg_profit"])) * 1.30
ax.set_ylim(-ymax, ymax * 0.45)
fig.tight_layout()
fig.savefig(FIG / "fig1_q1_bars.png")
plt.show()

## 2. By transaction cost type × cost region

In [ ]:
matrix = (pp.groupby(["transaction_cost_type", "cost_region"], as_index=False)
            .agg(n=("id", "count"),
                 avg_profit=("profit_gbp", "mean"),
                 total_profit=("profit_gbp", "sum"))
            .round(3))

# Pivot for readability — cost_type rows × cost_region columns
pivot_avg = (matrix.pivot(index="transaction_cost_type", columns="cost_region",
                          values="avg_profit")
                    .reindex(columns=["Domestic", "Intra", "Inter"])
                    .round(3))
pivot_n = (matrix.pivot(index="transaction_cost_type", columns="cost_region",
                        values="n")
                  .reindex(columns=["Domestic", "Intra", "Inter"])
                  .astype(int))
print("Avg profit per transaction (£):")
print(pivot_avg.to_string())
print("\nTransaction count (n):")
print(pivot_n.to_string())

In [ ]:
matrix = con.execute('''
    SELECT transaction_cost_type AS cost_type, cost_region,
           AVG(profit_gbp) AS avg_profit, COUNT(*) AS n
    FROM profit_per_transaction
    GROUP BY transaction_cost_type, cost_region
''').df()

regions = ["Domestic", "Intra", "Inter"]
ctypes  = ["POS/ECOM", "ATM"]
ctype_colors = {"POS/ECOM": BRIGHT_GREEN, "ATM": DARK_GREEN}

fig, ax = plt.subplots(figsize=(8.5, 4.8))
x = np.arange(len(regions))
bar_width = 0.36
ymin_data = matrix["avg_profit"].min()
ymax_data = matrix["avg_profit"].max()
for i, ct in enumerate(ctypes):
    sub = (matrix[matrix["cost_type"] == ct]
           .set_index("cost_region").reindex(regions).reset_index())
    offsets = x + (i - 0.5) * bar_width
    ax.bar(offsets, sub["avg_profit"], width=bar_width,
           color=ctype_colors[ct], edgecolor="none", label=ct, zorder=3)
    for xv, v in zip(offsets, sub["avg_profit"]):
        if pd.isna(v):
            continue
        ax.text(xv, v + (0.10 if v > 0 else -0.10),
                f"£{v:+.2f}", ha="center",
                va="bottom" if v > 0 else "top",
                fontsize=10, fontweight="bold", color=DARK_GREEN)
style_axes(ax, baseline=True)
ax.set_xticks(x); ax.set_xticklabels(regions, fontsize=11, fontweight="bold")
ax.set_xlabel("Cost region")
ax.set_ylabel("Avg profit per transaction (£)")
ax.set_title("Avg profit by cost type × cost region")
ax.set_ylim(ymin_data * 1.25, max(ymax_data * 1.8, 0.4))
ax.legend(loc="lower left", bbox_to_anchor=(0.0, -0.02))
fig.tight_layout()
fig.savefig(FIG / "fig1b_q1_matrix.png")
plt.show()

## 3. Driver decomposition — revenue vs cost components

In [ ]:
drivers = (con.execute('''
    SELECT transaction_type,
           AVG(interchange_revenue_gbp) AS interchange,
           AVG(conversion_revenue_gbp)  AS conversion,
           AVG(fixed_cost_gbp)          AS fixed_cost,
           AVG(variable_cost_gbp)       AS variable_cost,
           AVG(profit_gbp)              AS net_profit
    FROM profit_per_transaction
    GROUP BY transaction_type
''').df())
order = ["ECOM_PURCHASE", "POS_PURCHASE", "CASH_WITHDRAWAL"]
drivers = drivers.set_index("transaction_type").reindex(order).reset_index()

cols = ["Interchange", "Conversion", "Fixed cost", "Variable cost", "Net profit"]
rows = []
for _, r in drivers.iterrows():
    rows.append([
        r["transaction_type"],
        f"£{r['interchange']:+.4f}",
        f"£{r['conversion']:+.4f}",
        f"£{-r['fixed_cost']:+.4f}",
        f"£{-r['variable_cost']:+.4f}",
        f"£{r['net_profit']:+.4f}",
    ])

header = ["Transaction type"] + cols
widths = [max(len(str(h)), max(len(r[i]) for r in rows)) for i, h in enumerate(header)]
def fmt_row(vals):
    return "  ".join(str(v).ljust(widths[i]) if i == 0 else str(v).rjust(widths[i])
                     for i, v in enumerate(vals))
print(fmt_row(header))
print("  ".join("-" * w for w in widths))
for r in rows:
    print(fmt_row(r))

fig, ax = plt.subplots(figsize=(9.5, 2.4))
ax.axis("off")
table = ax.table(
    cellText=[r[1:] for r in rows],
    rowLabels=[r[0] for r in rows],
    colLabels=cols,
    cellLoc="right", rowLoc="left", colLoc="center", loc="center",
)
table.auto_set_font_size(False); table.set_fontsize(11); table.scale(1.0, 1.6)
for (row, col), cell in table.get_celld().items():
    cell.set_edgecolor("white"); cell.set_linewidth(1.2)
    if row == 0:
        cell.set_facecolor(DARK_GREEN); cell.set_text_props(color="white", fontweight="bold")
    elif col == -1:
        cell.set_facecolor(SOFT_GREEN); cell.set_text_props(fontweight="bold", color=DARK_GREEN)
    else:
        cell.set_facecolor("#F4F8F5" if row % 2 else "white")
        txt = cell.get_text().get_text()
        if txt.startswith("£-"):
            cell.set_text_props(color=DARK_GREEN)
        elif txt.startswith("£+"):
            cell.set_text_props(color=BRIGHT_GREEN, fontweight="bold")
ax.set_title("P&L decomposition per transaction type  \u2022  £ per transaction",
             loc="left", fontweight="bold", color=DARK_GREEN, pad=14)
fig.tight_layout()
fig.savefig(FIG / "fig2_q1_decomposition.png")
plt.show()